In [83]:
import numpy as np
import pandas as pd

data = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
data.drop(['track_id','album_name','track_name'], axis=1, inplace=True)
test.drop(['track_id','album_name','track_name'], axis=1, inplace=True)
data.head()

,artists,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,Tulipa Ruiz,39,172834,False,0.570,0.817,8,-4.657,1,0.0701,0.21400,0.0000,0.0771,0.776,174.241,3,mpb
1,Yanni,41,273640,False,0.303,0.135,3,-19.767,1,0.0308,0.94700,0.9440,0.0907,0.078,94.762,4,new-age
2,Paffendorf,28,212403,False,0.611,0.884,0,-6.360,0,0.0367,0.00093,0.2670,0.0889,0.772,139.001,4,techno
3,Floyd Cramer,17,135040,False,0.713,0.653,0,-13.365,1,0.0349,0.95000,0.8840,0.1030,0.921,152.758,4,honky-tonk
4,Dirt Monkey,24,1556883,False,0.528,0.780,2,-5.549,1,0.1210,0.02770,0.0168,0.1850,0.329,125.065,4,dub


In [84]:
data['explicit'] = data['explicit'].astype(int)
test['explicit'] = test['explicit'].astype(int)
data

,artists,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,Tulipa Ruiz,39,172834,0,0.570,0.817,8,-4.657,1,0.0701,0.214000,0.000000,0.0771,0.776,174.241,3,mpb
1,Yanni,41,273640,0,0.303,0.135,3,-19.767,1,0.0308,0.947000,0.944000,0.0907,0.078,94.762,4,new-age
2,Paffendorf,28,212403,0,0.611,0.884,0,-6.360,0,0.0367,0.000930,0.267000,0.0889,0.772,139.001,4,techno
3,Floyd Cramer,17,135040,0,0.713,0.653,0,-13.365,1,0.0349,0.950000,0.884000,0.1030,0.921,152.758,4,honky-tonk
4,Dirt Monkey,24,1556883,0,0.528,0.780,2,-5.549,1,0.1210,0.027700,0.016800,0.1850,0.329,125.065,4,dub
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85495,HELLYEAH,23,197601,1,0.485,0.984,7,-4.448,1,0.1190,0.000092,0.000002,0.1980,0.115,106.069,3,industrial
85496,Soda Stereo,41,363360,0,0.634,0.665,7,-12.186,1,0.0363,0.001190,0.579000,0.1480,0.206,113.966,4,punk
85497,Perfume,21,266506,0,0.659,0.961,0,-2.476,1,0.0385,0.019600,0.012600,0.1380,0.537,135.033,4,j-idol
85498,Nat King Cole,0,192160,0,0.320,0.210,6,-15.231,1,0.0345,0.920000,0.000080,0.1380,0.208,73.196,5,jazz


In [85]:
global_mean = data['popularity'].mean()
# global_mean1 = test['popularity'].mean()
print(f"The global mean popularity is: {global_mean:.2f}")

The global mean popularity is: 33.24


In [86]:
artist_means = data.groupby('artists')['popularity'].mean()
genre_means = data.groupby('track_genre')['popularity'].mean()

# artist_means1 = test.groupby('artists')['popularity'].mean()
# genre_means1 = test.groupby('track_genre')['popularity'].mean()
print("Artist-specific means calculated successfully. Displaying the head of artist_means:")
print(artist_means.head())

Artist-specific means calculated successfully. Displaying the head of artist_means:
artists
!nvite                                                23.000000
"Puppy Dog Pals" Cast                                 57.500000
"Weird Al" Yankovic                                   26.769231
#Kids;Nursery Rhymes;Nursery Rhymes and Kids Songs    34.000000
$affie                                                39.000000
Name: popularity, dtype: float64


In [87]:
m = 10
print(f"Smoothing factor (m) set to: {m}")

Smoothing factor (m) set to: 10


In [88]:
artist_counts = data.groupby('artists').size()
track_counts = data.groupby('track_genre').size()

# artist_counts1 = test.groupby('artists').size()
# track_counts1 = test.groupby('track_genre').size()

print("Artist counts calculated successfully. Displaying the head of artist_counts:")
print(artist_counts.head())

Artist counts calculated successfully. Displaying the head of artist_counts:
artists
!nvite                                                 2
"Puppy Dog Pals" Cast                                  2
"Weird Al" Yankovic                                   13
#Kids;Nursery Rhymes;Nursery Rhymes and Kids Songs     1
$affie                                                 1
dtype: int64


In [89]:
def smoothed_target_encode(artist_means, artist_counts, global_mean, m):
    return (artist_means * artist_counts + global_mean * m) / (artist_counts + m)

smoothed_artist_popularity = smoothed_target_encode(artist_means, artist_counts, global_mean, m)
smoothed_track_genre_popularity = smoothed_target_encode(genre_means, track_counts, global_mean, m)

# smoothed_artist_popularity1 = smoothed_target_encode(artist_means1, artist_counts1, global_mean1, m)
# smoothed_track_genre_popularity1 = smoothed_target_encode(genre_means1, track_counts1, global_mean1, m)
print("Smoothed artist popularity calculated successfully. Displaying the head of smoothed_artist_popularity:")
print(smoothed_artist_popularity.head())

Smoothed artist popularity calculated successfully. Displaying the head of smoothed_artist_popularity:
artists
!nvite                                                31.529873
"Puppy Dog Pals" Cast                                 37.279873
"Weird Al" Yankovic                                   29.580803
#Kids;Nursery Rhymes;Nursery Rhymes and Kids Songs    33.305316
$affie                                                33.759862
dtype: float64


In [90]:
data['artists'] = data['artists'].map(smoothed_artist_popularity)
data['track_genre'] = data['track_genre'].map(smoothed_track_genre_popularity)

# test['artists'] = test['artists'].map(smoothed_artist_popularity1)
# test['track_genre'] = test['track_genre'].map(smoothed_track_genre_popularity1)

print("Original 'artists' column replaced with smoothed target encoded values. Displaying the head of the updated DataFrame:")
data.head()

Original 'artists' column replaced with smoothed target encoded values. Displaying the head of the updated DataFrame:


,artists,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,33.759862,39,172834,0,0.570,0.817,8,-4.657,1,0.0701,0.21400,0.0000,0.0771,0.776,174.241,3,40.625138
1,36.485420,41,273640,0,0.303,0.135,3,-19.767,1,0.0308,0.94700,0.9440,0.0907,0.078,94.762,4,29.460102
2,32.759862,28,212403,0,0.611,0.884,0,-6.360,0,0.0367,0.00093,0.2670,0.0889,0.772,139.001,4,38.734627
3,21.310662,17,135040,0,0.713,0.653,0,-13.365,1,0.0349,0.95000,0.8840,0.1030,0.921,152.758,4,16.482313
4,31.529873,24,1556883,0,0.528,0.780,2,-5.549,1,0.1210,0.02770,0.0168,0.1850,0.329,125.065,4,39.682949


In [82]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

model = RandomForestRegressor(n_estimators=100, random_state=42)


X = data.drop('popularity', axis=1)
y = data['popularity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(model.score(X_test, y_test))


0.7463351008426793
